### Data structure


In [1]:
import os
import json
from pathlib import Path
from typing import List

from langchain_core.documents import Document

### JSON Loader


In [2]:
class JSONKnowledgeLoader:
    """
    Loads V2 Knowledge Cards from JSON files and converts
    them into LangChain Documents.
    """

    def __init__(self, data_directory: str):
        self.data_directory = data_directory

    def load(self) -> List[Document]:

        documents = []

        json_files = list(Path(self.data_directory).rglob("*.json"))

        print(f"Found {len(json_files)} JSON files.")

        for file in json_files:

            with open(file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Handle one card per file
            if isinstance(data, dict):
                data = [data]

            for card in data:

                searchable_text = self._create_searchable_text(card)

                documents.append(
                    Document(
                        page_content=searchable_text,
                        metadata={
                            "concept_id": card.get("concept_id"),
                            "concept_type": card.get("concept_type"),
                            "title": card.get("title"),
                            "source": str(file)
                        }
                    )
                )

        print(f"Loaded {len(documents)} knowledge cards.")

        return documents

    def _create_searchable_text(self, card):

        search = card.get("search", {})

        content = card.get("content", {})

        text = f"""
    Title:
    {card.get("title","")}

    Description:
    {card.get("description","")}

    Content:
    {json.dumps(content, indent=2)}

    Keywords:
    {' '.join(search.get("keywords",[]))}

    Aliases:
    {' '.join(search.get("aliases",[]))}

    User Queries:
    {' '.join(search.get("user_queries",[]))}
    """

        return text

loader = JSONKnowledgeLoader("../data/knowlege-cardV2/definitions")

documents = loader.load()
documents

Found 47 JSON files.
Loaded 47 knowledge cards.


[Document(metadata={'concept_id': 'definition.advertisement', 'concept_type': 'definition', 'title': 'Advertisement', 'source': '..\\data\\knowlege-cardV2\\definitions\\definition.advertisement.json'}, page_content='\n    Title:\n    Advertisement\n\n    Description:\n    Definition of Advertisement under the Consumer Protection Act, 2019\n\n    Content:\n    {\n  "term": "advertisement",\n  "legal_definition": "any audio or visual publicity, representation, endorsement or pronouncement made by means of light, sound, smoke, gas, print, electronic media, internet or website and includes any notice, circular, label, wrapper, invoice or such other documents",\n  "plain_language": "a public announcement or promotion using various media like audio, video, print, internet, etc.",\n  "examples": [\n    "A company running a TV commercial to promote its new product",\n    "A social media post endorsing a brand"\n  ],\n  "non_examples": [\n    "A private conversation between friends about a prod

### Embeddings


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\OMEN\Desktop\Codes\coding\python\RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
class EmbeddingManager:
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
            """
            Initialize the embedding model and vector database.

            Args:
            model_name: Name of the SentenceTransformer model.
            persist_directory: Folder where ChromaDB stores vectors.
            collection_name: Name of the vector collection.
            """
            self.model_name=model_name
            self.model=None
            self._load_model()
            # Load embedding model
            #self.model = SentenceTransformer(model_name)

            # Create ChromaDB client
            # self.client = chromadb.PersistentClient(
            #     path=persist_directory,
            #     settings=Settings(anonymized_telemetry=False)
            # )

            # # Create or load collection
            # self.collection = self.client.get_or_create_collection(
            #     name=collection_name
            # )

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed

        Returns:
            NumPy array of embeddings with shape
            (len(texts), embedding_dim)
            """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3322.44it/s]


Model loaded successfully. Embedding dimension: 384


### Vector DB


In [6]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store() 

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create ChromaDB persistent client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Create or load collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(f"Vector store initialized.")
            print(f"Collection: {self.collection.name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects
            embeddings: Corresponding embeddings
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)


            # Store metadata
            metadata = dict(doc.metadata) 
            metadata['content_length']=len(doc.page_content)
            metadata['doc_index'] = i

            metadatas.append(metadata)
            # Store document text
            documents_text.append(doc.page_content)

            # Convert numpy embedding to list
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents.")
            print(f"Total documents: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to collection .")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized.
Collection: pdf_documents
Existing documents: 939


In [7]:
texts=[doc.page_content for doc in documents ]

embeddings=embedding_manager.generate_embeddings(texts)

## store in db

vectorstore.add_documents(documents,embeddings)

Generating embeddings for 47 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.36it/s]


Generated embeddings with shape: (47, 384)
Adding 47 documents to vector store...
Successfully added 47 documents.
Total documents: 770


In [14]:
class RAGRetriever:
    """Handles query based retrieval from the vector store"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """Initialize the retreiver 
        
        Args 
        vector store : Vector store containing the document embeddings
        embedding_manager: Manager for generating query embeddings"""

        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
            """
            Retrieve relevant documents for a query
            
            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold
                
            Returns:
                List of dictionaries containing retrieved documents and metadata
            """
            print(f"Retrieving documents for query: '{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")
            
            # Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
            
            # Search in vector store
            try:
                results = self.vector_store.collection.query(
                    query_embeddings=[query_embedding.tolist()],
                    n_results=top_k
                )
                
                # Process results
                retrieved_docs = []
                
                if results['documents'] and results['documents'][0]:
                    documents = results['documents'][0]
                    metadatas = results['metadatas'][0]
                    distances = results['distances'][0]
                    ids = results['ids'][0]
                    
                    for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                        # Convert distance to similarity score (ChromaDB uses cosine distance)
                        similarity_score = 1 - distance
                        
                        if similarity_score >= score_threshold:
                            retrieved_docs.append({
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i + 1
                            })
                    
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                else:
                    print("No documents found")
                
                return retrieved_docs
                
            except Exception as e:
                print(f"Error during retrieval: {e}")
                return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever
result=rag_retriever.retrieve("what is a consumer")
print(result)


Retrieving documents for query: 'what is a consumer'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
[{'id': 'doc_450eee5b_9', 'content': '(8) "consumer dispute" means a dispute where the person against whom a\ncomplaint has been made, denies or disputes the allegations contained in the complaint;\n(9) "consumer rights" includes,—\n(i) the right to be protected against the marketing of goods, products or\nservices which are hazardous to life and property;\n(ii) the right to be informed about the quality, quantity, potency, purity,\nstandard and price of goods, products or services, as the case may be, so as to\nprotect the consumer against unfair trade practices;', 'metadata': {'content_length': 525, 'title': '2606GI.p65', 'creationDate': "D:20190809230405+05'30'", 'keywords': '', 'creator': 'PageMaker 6.5', 'page': 2, 'total_pages': 40, 'format': 'PDF 1.6', 'source': '..\\data\\Consumer Protection Act, 2019.pdf', 'author': 'Administrator', 'doc_index': 9, 'trapped': '', 'creationdate': '2019-08-09T23:04

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from google import genai
import os


load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key,
    temperature=0.3,
)


def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [9]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [10]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is a consumer ", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is a consumer '
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
(8) "consumer dispute" means a dispute where the person against whom a
complaint has been made, denies or disputes the allegations contained in the complaint;
(9) "consume

r rights" includes,—
(i) the right to be protected against the marketing of goods, products or
services which are hazardous to life and property;
(ii) the right to be informed about the quality, quantity, potency, purity,
standard and price of goods, products or services, as the case may be, so as to
protect the consumer against unfair trade practices;

(8) "consumer dispute" means a dispute where the person against whom a
complaint has been made, denies or disputes the allegations contained in the complaint;
(9) "consumer rights" includes,—
(i) the right to be protected against the marketing of goods, products or
services which are hazardous to life and property;
(ii) the right to be informed about the quality, quantity, potency, purity,
standard and price of goods, products or services, as the case may be, so as to
protect the consumer against unfair trade practices;

(8) "consumer dispute" means a dispute where the person against whom a
complaint has been made, denies or disputes th

TypeError: can only concatenate list (not "str") to list